# Monte Carlo Pricing and Interest Rate Swaps

Simulation of vanilla and barrier derivatives, antithetic variates, swap valuation and stochastic-calculus exercises.

In [ ]:
import numpy as np
from bsm import BSM

### Monte Carlo Pricing and Variance Reduction

Consider various Put options on oil:

1. Plain-vanilla option with strike $K=\$60$ paying $(K-F_T)^{+}$ USD at maturity. $F_T$ is the oil future price at $T$.

2. Barrier option with strike $K=\$60$ and knock-out barrier of $B=\$45$.



Oil current future price is $F_0 = \$70$. 

It follows the process $dF = F \sigma_F dB$ in risk neutral USD measure.


The risk free interest rate is 2% (continuous compounding).

$\sigma_{F} = 45\%$ 
    
Run 100,000 MC simulations.

 a. Realize MC without variance reduction (VR) to price these options (price and MC error)

1. Divide the path into $m$ time steps $Δt = T/m$ , and simulate $n$ sample paths of the risk neutral price process. 

2. Calculate the terminal payoff for each path. The payoff on the ith path:

  $\{s^i_{k},\;k=1,..,m \}$  is

  $f^{i}_{0} = F(s^i_{0},..,s^i_{k},..s^i_{m})$

3. A Monte Carlo estimate of the security price is just the discounted average of all $n$ payoffs, one for each sample path you generated: 

    $\hat{f_{0}} = e^{-rT} \frac{\sum f^i}{ n}$

The futures price follows a zero-drift risk-neutral geometric Brownian motion: $$F_{t_k}=F_{t_{k-1}}\exp\left(-\frac{\sigma^2}{2}\Delta t+\sigma\sqrt{\Delta t}\,\epsilon_k\right).$$

In [ ]:
np.random.seed(43)
F_0, r, q, vol = 70, 0.02, 0.02, 0.45  # r=q: futures have zero risk-neutral drift
K, T, m = 60, 1, 252
B = 45
N = 10**5

In [ ]:
#Plain-vanilla Put option
dt = T / m
drift = ( - 0.5 * vol**2) * dt
diffusion = vol * np.sqrt(dt)

Z = np.random.normal(0, 1, (N, m)) 

S_k = np.zeros((N, m + 1))
S_k[:, 0] = F_0  

for k in range(1, m + 1):
    log_F = drift + diffusion * Z[:, k - 1]
    S_k[:, k] = S_k[:, k - 1] * np.exp(log_F)

payoffs = np.maximum(K - S_k[:, -1], 0)

price_plain = np.exp(-r * T) * np.mean(payoffs)
std_plain = np.exp(-r * T) * np.std(payoffs, ddof=1) / np.sqrt(N)

print(f"Plain-vanilla Put option price: {price_plain} USD and MC error: {std_plain}")   

In [ ]:
#Barrier Put option
dt = T / m
drift = ( - 0.5 * vol**2) * dt
diffusion = vol * np.sqrt(dt)

Z = np.random.normal(0, 1, (N, m)) 

S_k = np.zeros((N, m + 1)) 
S_k[:, 0] = F_0 

for k in range(1, m + 1):
    log_F = drift + diffusion * Z[:, k - 1]
    S_k[:, k] = S_k[:, k - 1] * np.exp(log_F)

knock_out = np.any(S_k[:, 1:] <= B, axis = 1)

payoffs = np.zeros(N)
payoffs[~knock_out] = np.maximum(K - S_k[~knock_out, -1], 0)

price_barrier = np.exp(-r * T) * np.mean(payoffs)
std_barrier = np.exp(-r * T) * np.std(payoffs, ddof=1) / np.sqrt(N)

print(f"Barrier option price: {price_barrier} USD and MC error: {std_barrier}")

b.  Realize MC with AV in all case. What is the reduction in MC error?

In [ ]:
#Plain-vanilla Put option
dt = T / m
drift = ( - 0.5 * vol**2) * dt
diffusion = vol * np.sqrt(dt)

Z = np.random.normal(0, 1, (N, m)) 
Z_neg = -Z

S_k_pos = np.zeros((N, m + 1))
S_k_pos[:, 0] = F_0  

S_k_neg = np.zeros((N, m + 1))
S_k_neg[:, 0] = F_0  

for k in range(1, m + 1):
    log_F_pos = drift + diffusion * Z[:, k - 1]
    S_k_pos[:, k] = S_k_pos[:, k - 1] * np.exp(log_F_pos)
    
    log_F_neg = drift + diffusion * Z_neg[:, k - 1]
    S_k_neg[:, k] = S_k_neg[:, k - 1] * np.exp(log_F_neg)
    
payoff_pos = np.maximum(K - S_k_pos[:, -1], 0)
payoff_neg = np.maximum(K - S_k_neg[:, -1], 0)

payoffs = 0.5 * (payoff_pos + payoff_neg)

price_plain_AV = np.exp(-r * T) * np.mean(payoffs)
std_plain_AV = np.exp(-r * T) * np.std(payoffs, ddof=1) / np.sqrt(N)

print(f"Plain-vanilla Put option price: {price_plain_AV} USD and MC error: {std_plain_AV}")  

In [ ]:
#Barrier Put option
dt = T / m
drift = ( - 0.5 * vol**2) * dt
diffusion = vol * np.sqrt(dt)

Z = np.random.normal(0, 1, (N, m)) 
Z_neg = -Z

S_k_pos = np.zeros((N, m + 1))
S_k_pos[:, 0] = F_0  

S_k_neg = np.zeros((N, m + 1))
S_k_neg[:, 0] = F_0  

for k in range(1, m + 1):
    log_F_pos = drift + diffusion * Z[:, k - 1]
    S_k_pos[:, k] = S_k_pos[:, k - 1] * np.exp(log_F_pos)
    
    log_F_neg = drift + diffusion * Z_neg[:, k - 1]
    S_k_neg[:, k] = S_k_neg[:, k - 1] * np.exp(log_F_neg)

knock_out_pos = np.any(S_k_pos[:, 1:] <= B, axis = 1)
knock_out_neg = np.any(S_k_neg[:, 1:] <= B, axis = 1)

payoffs_pos = np.zeros(N)
payoffs_pos[~knock_out_pos] = np.maximum(K - S_k_pos[~knock_out_pos, -1], 0)

payoffs_neg = np.zeros(N)
payoffs_neg[~knock_out_neg] = np.maximum(K - S_k_neg[~knock_out_neg, -1], 0)

payoffs = 0.5 * (payoffs_pos + payoffs_neg)

price_barrier_AV = np.exp(-r * T) * np.mean(payoffs)
std_barrier_AV = np.exp(-r * T) * np.std(payoffs, ddof=1) / np.sqrt(N)

print(f"Barrier option price: {price_barrier_AV} USD and MC error: {std_barrier_AV}")

In [ ]:
print(f"Reduction in MC error for plain-vanilla = {100 * (std_plain - std_plain_AV) / std_plain} %")
print(f"Reduction in MC error for barrier = {100 * (std_barrier - std_barrier_AV) / std_barrier} %")

c. Calculate the prices of the options the analytical formulars. Compare the results with MC.

In [ ]:
#we will calculate prices analytically using BSM module, as done in the earlier pricing exercise

In [ ]:
# 1. Closed-form Black--Scholes price of the vanilla put on futures.
a_vanilla = BSM(F_0, vol, r, q)  # r=q for a futures underlying
Put_vanilla = a_vanilla.Put(K, T)
print(f"Analytical vanilla put price: {Put_vanilla:.5f} USD")

# 2. Analytical continuously monitored down-and-out put.
# Use the absorbing log-price transition density (reflection principle).
from scipy.integrate import quad
from scipy.stats import norm

x0 = np.log(F_0 / B)
mu = -0.5 * vol**2
sd = vol * np.sqrt(T)
reflection = np.exp(-2 * mu * x0 / vol**2)

def absorbed_density(y):
    return (norm.pdf((y - x0 - mu*T) / sd)
            - reflection * norm.pdf((y + x0 - mu*T) / sd)) / sd

barrier_analytic = np.exp(-r*T) * quad(
    lambda y: (K - B*np.exp(y)) * absorbed_density(y),
    0, np.log(K/B)
)[0]
print(f"Continuous-monitoring down-and-out put: {barrier_analytic:.5f} USD")
print("Note: the Monte Carlo barrier is monitored at discrete daily intervals;")
print("its price may differ slightly from the continuously monitored benchmark.")

The futures process has **zero risk-neutral drift**, as specified in the problem. In the Black–Scholes implementation this corresponds to setting dividend yield $q=r$, not to adding an $r$ drift to simulated futures prices. The continuously monitored barrier benchmark can differ from a daily-monitored Monte Carlo price because paths may cross the barrier between observation dates.

The analytical vanilla price and the Monte Carlo estimate are based on the same risk-neutral futures model; any remaining difference should be evaluated relative to the Monte Carlo standard error.

### Interest Rate Swaps and Forward Bonds

In [ ]:
times = [0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5]
dfs = [.9748, .9494, .9238, .8982, .8725, .8470, .8216, .7965, .7717, .7472]
N = 10

a. Consider IRS swap with notional $N=10$ m USD. Calculate its fair swap rate.


A the swap rate is found to be:

$s = \frac{1-P_0(t_n)}{\tau \sum \limits_{i=1}^{n} P_0(t_i)} $

In [ ]:
s = (1 - dfs[-1]) / (0.5 * sum(dfs))
print(f'Fair swap rate = {s * 100} %')

b. In one year continuously compounded yield−to−maturity rates went up by 1%. Mark-to-market this swap.

To mark the swap to market one year later, I infer spot rates from the original discount factors, shock the rates by one percentage point, then value the remaining cash flows. The swap has four years of remaining maturity.

In [ ]:
# Infer original continuously compounded spot rates and apply a +1 pp shock.
times = np.asarray(times, dtype=float)
dfs = np.asarray(dfs, dtype=float)
y_old = -np.log(dfs) / times
y_new = y_old + 0.01

In [ ]:
# Interpret the shocked curve at t=1, over the remaining payment dates.
remaining_dates = times[times > 1]
times_remaining = remaining_dates - 1
# Use the shifted original zero-rate observations for remaining tenors.
y_remaining = y_old[:len(remaining_dates)] + 0.01
dfs_new = np.exp(-y_remaining * times_remaining)

In [ ]:
dfs_new

For a receive-floating, pay-fixed swap, the value at the valuation date is

$$V_{\mathrm{swap}}=N(1-P(t,T_n))-s\tau N\sum_{T_i>t}P(t,T_i),$$

where the remaining discount factors are measured **from the new valuation date** $t=1$ and the original fixed rate $s$ is retained.

In [ ]:
# Remaining semiannual payments are due at t=1.5, ..., 5.0 years.
# Value of receive-floating / pay-fixed swap at the new valuation date.
V_swap = N * (1 - dfs_new[-1]) - s * 0.5 * N * np.sum(dfs_new)
print(f'Swap value after the rate shock: {V_swap:.4f} million USD')

c. 

Consider that you are a bank. A client requested the price of the forward bond for the period from t=2Y to t=4Y. According the forward bond agreement:

1. the client pays the bond price $B_f$ at time $t=2Y$

2. the client receives the bond notional amont $N = 1,000$ at time $t=4Y$. No coupons are paid (this is a 2 year discound bond). 

3. Price $B_f$ is fixed now at $t=0$

Quote fair $B_f$. 

Fair $B_f$ implies that the value of the forward argeement is equal to zero at t=0.

The forward bond has the following cash flows:

|  <font size=4>__t = 1__</font> |  <font size=4>t = 2</font> |  <font size=4>t = 3</font> | <font size=4>t = 4</font> |
| :--------  | :---: |  :---: | :---: |
| <font size=3> 0  </font>   | <font size=3>    -$B_f$ </font> | <font size=3>    0 </font> | <font size=3>    N  </font> |

At a fair forward price $B_f$, $ - P_0(2) B_f + P_0(4) N = 0$

In [ ]:
# use the original discount factors; the rate shock applies only to part (b)
N = 1000
B_f = N * dfs[7] / dfs[3]

print(f'Fair B_f = {B_f}')

### Itô’s Lemma and Stochastic Processes

Recall Itô's lemma:

$dX_t = a(X_t,t)dt + b(X_t,t)dB_t$

Let $f(x,t)$ be a twice continuously differentiable function of two variables and define a
new process $\{f_t := f(X_t,t), t≥0\}$.

$df_t =  \left(\frac{\partial f}{\partial t} + a(.) \frac{\partial f}{\partial x}+\frac{1}{2} b^2(.) \frac{\partial^2 f}{\partial X^2}\right)dt + b(.) \frac{\partial f}{\partial x}dB$


a) $~~Y/X$ exchange rate $S_t$ follows the process

$$S_t = (r-q)S_tdt + \sigma S_t dB_t$$

Using Ito's lemma, show that inverse exhange rate $1/S_t$ has the same vol.

here $a = (r - q) S_t$ and b = $\sigma S_t$

$f(S_t) = \frac{1}{S_t}$

$df = (\frac{df}{dt} + a(.) \frac{df}{dS_t} + 0.5 b^2(.) \frac{d^2f}{d^2S_t}) dt + b(.)\frac{df}{dS}dB = (0 + (r - q) S_t (- \frac{1}{S_t^2}) + 0.5 * \sigma^2 S^2_t \frac{2}{S^3_t})dt + \sigma S_t (- \frac{1}{S_t^2}) dB$

$df = ( (r - q) (- \frac{1}{S_t}) +  \sigma^2 \frac{1}{S_t}) dt + \sigma (- \frac{1}{S_t}) dB = \frac{\sigma^2 - (r - q)}{S_t} dt - \frac{\sigma}{S_t} dB$

$df_t = (\sigma^2 - (r-q))f_t\,dt - \sigma f_t\,dB_t$

The final equation shows directly that $vol(f_t) = \sigma$ - the same as for the original exchange rate

b) Using Ito's lemma, derive the process for $B^{2}_t$.

here $a = 0 $ and b = 1$

$f(B_t) = B^2_t$

$df = (\frac{df}{dt} + a(.) \frac{df}{dB_t} + 0.5 b^2(.) \frac{d^2f}{d^2B_t}) dt + b(.)\frac{df}{dB_t}dB = (0 +  0.5 * 2) dt + 2 B_t dB = dt + 2 B_t dB $